# Inverter VTC and ring-oscillator frequency, from four constants

Companion to web-app testbenches **26 “CMOS inverter VTC”** and
**29 “Ring oscillator”**. These use the *square-law* `nmos`/`pmos` models
($K_p$, $W/L$, $V_{th}$, λ — nothing else), which makes them the mirror
image of notebook 05: there, pen-and-paper chased a real BSIM4 device and
the $g_m/I_D$ lookup won; here the textbook math should be **essentially
exact**, because the textbook math *is* the device model. Everything below
is derived from four constants and checked against the simulator.

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt

from photonflux.nb import Session, Builder

s = Session()
vtc_bench = s.load_example("26_cmos_inverter_vtc")
mn = vtc_bench.instances["MN"]["settings"]
mp = vtc_bench.instances["MP"]["settings"]
VDD = vtc_bench["VDD.V"]
BN = mn["Kp"] * mn["W"] / mn["L"]              # beta_n [A/V^2]
VTN, VTP = mn["Vth"], mp["Vth"]
print(f"NMOS: Kp={mn['Kp']:g}, W/L={mn['W'] / mn['L']:g}, Vth={VTN} | "
      f"PMOS: Kp={mp['Kp']:g}, W/L={mp['W'] / mp['L']:g}, Vth={VTP} | "
      f"VDD={VDD}")

## 0. Calibrate: is the model really $I_D = \frac{K_p}{2}\frac{W}{L}
(V_{GS}-V_{th})^2$?

Trust nothing — one Builder testbench (transistor + 1 Ω current-sense
resistor, built from the notebook) pins the convention, including the
factor of ½, before we lean on it.

In [ ]:
b = Builder()
m = b.add("nmos", ref="MN1", Kp=mn["Kp"], W=mn["W"], L=mn["L"],
          Vth=VTN, lam=0.0)
vg = b.add("vdc", ref="VG1", V=VDD)
vd = b.add("vdc", ref="VD1", V=VDD)
rs = b.add("resistor", ref="RS1", R=1.0)
g1, g2, g3 = (b.add("ground") for _ in range(3))
b.wire(vd.p1, m.d), b.wire(vg.p1, m.g), b.wire(m.s, rs.p1)
b.wire(rs.p2, g1.p1), b.wire(vg.p2, g2.p1), b.wire(vd.p2, g3.p1)
b.probe(m.s, "id_sense")
cal = s.dcsweep("VG1", "V", VTN, VDD, points=24, schematic=b)
id_meas = cal["id_sense"]                       # 1 Ω sense: volts == amps
id_theory = BN / 2 * np.maximum(cal.x - VTN, 0) ** 2
dev = np.abs(id_meas[5:] / id_theory[5:] - 1).max()
print(f"Id at Vgs={VDD} V: {id_meas[-1] * 1e3:.4f} mA "
      f"(Kp/2·W/L·(Vgs−Vth)² = {id_theory[-1] * 1e3:.4f} mA), "
      f"max dev {dev * 100:.2f}%")
assert dev < 0.01, "square-law convention mismatch"

## 1. The switching threshold $V_M$

At $V_M$ both devices saturate and their currents must match:

$$ V_M = \frac{V_{tn} + r\,(V_{DD} - |V_{tp}|)}{1 + r}, \qquad
   r = \sqrt{\beta_p / \beta_n} $$

The stored analysis steps the PMOS width over 10/20/40 µm, sliding $r$
through $1/\sqrt2$, 1, $\sqrt2$ — the classic skew study. (λ = 0.05/V
nudges the exact crossing by a few mV; the λ-free formula is the point.)

In [ ]:
res = s.run(schematic=vtc_bench)
vin = res.x
fig, ax = plt.subplots(figsize=(7, 4))
print(f"{'Wp':>6s} {'r':>6s} {'VM hand':>8s} {'VM sim':>7s} {'gain':>6s}")
for name, vout in res.family("vout").items():
    wp = float(name.split("@")[1].strip().rstrip("V"))
    r = np.sqrt(mp["Kp"] * wp / mp["L"] / BN)
    vm_hand = (VTN + r * (VDD - abs(VTP))) / (1 + r)
    d = vout - vin                              # crossing vout = vin
    k = int(np.where(np.diff(np.sign(d)))[0][0])
    vm_sim = float(vin[k] - d[k] * (vin[k + 1] - vin[k]) / (d[k + 1] - d[k]))
    gain = float(np.abs(np.gradient(vout, vin)).max())
    ax.plot(vin, vout, label=f"Wp = {wp * 1e6:.0f} µm: "
            f"$V_M$ = {vm_sim:.3f} (hand {vm_hand:.3f})")
    ax.plot([vm_hand], [vm_hand], "kx", ms=7)
    print(f"{wp * 1e6:4.0f}µm {r:6.3f} {vm_hand:8.3f} {vm_sim:7.3f} {gain:6.1f}")
    assert abs(vm_sim - vm_hand) < 0.03, (wp, vm_sim, vm_hand)
ax.plot(vin, vin, ":", color="gray", lw=.8)
ax.set_xlabel("Vin [V]"), ax.set_ylabel("Vout [V]")
ax.legend(fontsize=8), ax.grid(alpha=.3)
ax.set_title("VTC vs the saturation-current-balance $V_M$ (×)")
fig.tight_layout()

## 2. Ring-oscillator frequency, two grades of theory

Testbench 29 chains five of these inverters with explicit 30 fF loads
(the square-law devices carry no capacitance of their own — the delay is
*all* $C\,dV/I$, which we confirm below by the frequency scaling exactly
as $1/C$). Grade one is the classic back-of-envelope: gate stepped to
$V_{DD}$, discharge in two phases,

* saturation from $V_{DD}$ down to $V_{ov} = V_{DD}-V_{tn}$:
  $t_1 = C\,(V_{DD}-V_{ov})/I_{dsat}$,
* triode from $V_{ov}$ to $V_{DD}/2$:
  $t_2 = \frac{C}{\beta_n V_{ov}} \ln\frac{2V_{ov} - V_{DD}/2}{V_{DD}/2}$,

with $f = 1/(2\sum_i t_{p,i})$ over the five stages (the kick-capacitor
stage carries 35 fF). **Expect this to come out ~2× fast** — and that
error is the interesting part, see below.

In [ ]:
osc_bench = s.load_example("29_ring_oscillator")
C_STAGE = osc_bench["C0.C"]
C_KICK = osc_bench["CK.C"]
N_STAGES = 5


def t_p(beta, vth, c):
    vov = VDD - vth
    i_sat = beta / 2 * vov ** 2
    t1 = c * (VDD - vov) / i_sat
    t2 = c / (beta * vov) * np.log((2 * vov - VDD / 2) / (VDD / 2))
    return t1 + t2


BP_OSC = (osc_bench["MP0.Kp"] * osc_bench["MP0.W"] / osc_bench["MP0.L"])
loads = np.array([C_STAGE + (C_KICK if i == 0 else 0)
                  for i in range(N_STAGES)])
period_step = sum(t_p(BN, VTN, c) + t_p(BP_OSC, abs(VTP), c) for c in loads)
f_step = 1 / period_step
print(f"t_p(30 fF, step input) = {t_p(BN, VTN, C_STAGE) * 1e12:.1f} ps  ->  "
      f"f_step = {f_step / 1e9:.3f} GHz")

Grade two costs ten lines of numpy instead of a napkin, but is *still*
pen-and-paper in spirit — the same four constants, no simulator: each node
obeys $C_i \dot V_i = I_P(V_{i-1}, V_i) - I_N(V_{i-1}, V_i)$, so integrate
the five coupled square-law ODEs with RK4. This is the honest hand model:
it keeps what the step estimate throws away — each stage is driven by its
predecessor's **ramp** (the NMOS spends the first stretch of every
transition barely on, at $V_{ov}$ far below $V_{DD}-V_{tn}$) and the
not-yet-off complementary device fights back (short-circuit current).

In [ ]:
def i_nmos(vg, vd, beta, vth):
    vov = vg - vth
    lin = beta * (vov * vd - 0.5 * vd ** 2)
    sat = beta / 2 * vov ** 2
    return np.where(vov <= 0, 0.0, np.where(vd < vov, lin, sat))


def ring_rhs(v):
    vin = np.roll(v, 1)
    i_n = i_nmos(vin, v, BN, VTN)
    i_p = i_nmos(VDD - vin, VDD - v, BP_OSC, abs(VTP))   # PMOS by symmetry
    return (i_p - i_n) / loads


def crossings(t, v, settle=20e-9):
    m = t > settle
    ts, vs = t[m], v[m]
    above = vs > VDD / 2
    r = np.where(~above[:-1] & above[1:])[0]
    return ts[r] + (VDD / 2 - vs[r]) / (vs[r + 1] - vs[r]) * (ts[r + 1] - ts[r])


dt = 1e-12
v = np.array([0.2, 3.1, 0.2, 3.1, 1.0])          # any asymmetric start
tr, vr = [], []
for k in range(int(50e-9 / dt)):
    k1 = ring_rhs(v)
    k2 = ring_rhs(v + dt / 2 * k1)
    k3 = ring_rhs(v + dt / 2 * k2)
    k4 = ring_rhs(v + dt * k3)
    v = v + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    if k % 10 == 0:
        tr.append(k * dt), vr.append(v[1])
f_ode = 1 / float(np.diff(crossings(np.array(tr), np.array(vr))).mean())
print(f"golden square-law ODE: f_ode = {f_ode / 1e9:.3f} GHz")

In [ ]:
osc = s.run(schematic=osc_bench)                 # stored 50 ns transient
t = osc.x
cross = crossings(t, osc["n1"])
f_sim = 1 / float(np.diff(cross).mean())
print(f"f_sim = {f_sim / 1e9:.3f} GHz | ODE {f_ode / 1e9:.3f} GHz "
      f"({(f_sim / f_ode - 1) * 100:+.1f}%) | step estimate "
      f"{f_step / 1e9:.3f} GHz (×{f_step / f_sim:.2f} fast)")

fig, ax = plt.subplots(figsize=(9, 3.4))
win = (t > 20e-9) & (t < 24e-9)
ax.plot(t[win] * 1e9, osc["n1"][win], label="n1 (simulator)")
ax.plot(t[win] * 1e9, osc["n3"][win], label="n3", alpha=.7)
ax.axhline(VDD / 2, color="gray", lw=.7, ls=":")
ax.set_xlabel("time [ns]"), ax.set_ylabel("V"), ax.legend(fontsize=8)
ax.set_title(f"five-stage ring: simulator {f_sim / 1e9:.3f} GHz, "
             f"four-constant ODE {f_ode / 1e9:.3f} GHz")
ax.grid(alpha=.3)
fig.tight_layout()
assert abs(f_sim / f_ode - 1) < 0.03, f_sim / f_ode
assert 1.5 < f_step / f_sim < 2.6, f_step / f_sim

The step-input formula is ×~2 optimistic — *not* because square-law math
fails, but because its input-is-a-step premise does: in a ring, each gate
is driven by the previous stage's equally-slow ramp, so the driver spends
most of the transition at a fraction of its full $V_{ov}$, and the
complementary device leaks short-circuit current on top. Feed the same
four constants into the actual coupled ODEs and the frequency lands within
a percent of the simulator. That's the real lesson of hand analysis:
know which *assumption*, not which equation, you're paying for.

---
**Takeaways.** With the square-law model the saturation-balance $V_M$
formula tracks the simulated crossing to millivolts across a 4× skew
sweep; the ring frequency needs the ramp-driven picture (golden ODE,
~1 %) while the step-input napkin lands the order of magnitude and a
lesson. Read together with notebook 05: hand analysis is exact when the
model is the textbook — and the assumptions, not the algebra, are what
break first.

**Things to try**

* Double every stage load to 60 fF in the browser — frequency should
  halve almost exactly (delay here is all explicit C).
* Skew the oscillator's PMOS widths and watch duty cycle and frequency
  move together; the $V_M$ formula predicts the direction.
* Testbenches 27/28/30/31 (NAND, SR latch, mux, Schmitt) are the same
  square-law devices — every one is hand-analyzable the same way.